In [121]:
import pandas as pd
import numpy as np

In [122]:
df = pd.read_excel('../data/raw/online Retail.xlsx')

In [123]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [124]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [125]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [126]:
df[df["Description"].isna()].head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom


In [127]:
df[df["Description"].isna()]["StockCode"].nunique()

960

We can track the missing description by using Stockcode

In [128]:
missing = df[df['Description'].isna()]

In [129]:
missing

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
535322,581199,84581,NaN,-2,2011-12-07 18:26:00,0.0,NaN,United Kingdom
535326,581203,23406,NaN,15,2011-12-07 18:31:00,0.0,NaN,United Kingdom
535332,581209,21620,NaN,6,2011-12-07 18:35:00,0.0,NaN,United Kingdom
536981,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom


In [130]:
notnull_des_stock_code = df[df['Description'].notna()]['StockCode']

In [131]:
notnull_des_stock_code.unique()

array(['85123A', 71053, '84406B', ..., '90214U', '47591b', 23843],
      shape=(3958,), dtype=object)

In [132]:
recoverable = missing["StockCode"].isin(notnull_des_stock_code)

In [133]:
recoverable

622        True
1970      False
1971       True
1972       True
1987      False
          ...  
535322     True
535326     True
535332     True
536981     True
538554     True
Name: StockCode, Length: 1454, dtype: bool

In [134]:
print("Missing descriptions:", len(missing))
print("Recoverable descriptions:", recoverable.sum())

Missing descriptions: 1454
Recoverable descriptions: 1342


In [135]:
valid = df[df['Description'].notna()]

In [136]:
lookup = valid[['Description','StockCode']].drop_duplicates('StockCode')

In [137]:
lookup

,Description,StockCode
0,WHITE HANGING HEART T-LIGHT HOLDER,85123A
1,WHITE METAL LANTERN,71053
2,CREAM CUPID HEARTS COAT HANGER,84406B
3,KNITTED UNION FLAG HOT WATER BOTTLE,84029G
4,RED WOOLLY HOTTIE WHITE HEART.,84029E
...,...,...
509369,GREEN BITTY LIGHT CHAIN,85179a
512588,SET 10 CARDS SWIRLY XMAS TREE 17104,23617
527065,"LETTER ""U"" BLING KEY RING",90214U
537224,SCOTTIES CHILDRENS APRON,47591b


In [138]:
for index in missing.index:
    stock = df.loc[index, "StockCode"]

    match = lookup[lookup["StockCode"] == stock]

    if len(match) > 0:
        df.loc[index, "Description"] = match.iloc[0]["Description"]

In [139]:
df["Description"].isna().sum()

np.int64(112)

In [140]:
remaining = df[df["Description"].isna()]
remaining[["StockCode", "Description"]].head(20)

,StockCode,Description
1970,21134,NaN
1987,85226A,NaN
1988,85044,NaN
2024,20950,NaN
2026,84670,NaN
7187,35951,NaN
7193,21653,NaN
19628,20849,NaN
19631,72803B,NaN
21782,85018C,NaN


In [141]:
remaining_stock_codes = remaining["StockCode"].unique()

In [142]:
remaining

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
280754,561498,21610,NaN,-14,2011-07-27 14:10:00,0.0,NaN,United Kingdom
281615,561555,37477B,NaN,-11,2011-07-28 10:21:00,0.0,NaN,United Kingdom
281616,561557,37477C,NaN,-31,2011-07-28 10:21:00,0.0,NaN,United Kingdom
346849,567207,35592T,NaN,4,2011-09-19 11:01:00,0.0,NaN,United Kingdom


In [143]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [144]:
remaining[["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "CustomerID", "Country"]].head(20)

,InvoiceNo,StockCode,Quantity,UnitPrice,CustomerID,Country
1970,536545,21134,1,0.0,NaN,United Kingdom
1987,536549,85226A,1,0.0,NaN,United Kingdom
1988,536550,85044,1,0.0,NaN,United Kingdom
2024,536552,20950,1,0.0,NaN,United Kingdom
2026,536554,84670,23,0.0,NaN,United Kingdom
7187,536995,35951,57,0.0,NaN,United Kingdom
7193,537001,21653,-6,0.0,NaN,United Kingdom
19628,537875,20849,1,0.0,NaN,United Kingdom
19631,537878,72803B,1,0.0,NaN,United Kingdom
21782,538133,85018C,3,0.0,NaN,United Kingdom


In [145]:
remove = df['Description'].isna() & (df['UnitPrice'] == 0)

In [146]:
df = df[remove == False]

In [147]:
df['Description'].isnull().sum()

np.int64(0)

## Removing cancelled row

In [148]:
cancelled = df["InvoiceNo"].astype(str).str.startswith("C")

In [149]:
print("cancelled products",cancelled.sum())

cancelled products 9288


In [150]:
remove = df["InvoiceNo"].astype(str).str.startswith("C")

In [151]:
df = df[remove == False]

In [152]:
df["InvoiceNo"].astype(str).str.startswith("C").any()

np.False_

In [153]:
remove = df[(df["Quantity"] == 0) | (df["Quantity"] < 0)]

In [154]:
len(remove)

1239

In [155]:
remove[["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "CustomerID"]].head(20)

,InvoiceNo,StockCode,Quantity,UnitPrice,CustomerID
2406,536589,21777,-10,0.0,NaN
4347,536764,84952C,-38,0.0,NaN
7188,536996,22712,-20,0.0,NaN
7189,536997,22028,-20,0.0,NaN
7190,536998,85067,-6,0.0,NaN
7192,537000,21414,-22,0.0,NaN
7195,537003,85126,-2,0.0,NaN
7196,537004,21814,-30,0.0,NaN
7197,537005,21692,-70,0.0,NaN
7198,537006,21648,-130,0.0,NaN


In [156]:
remove = df["Quantity"] <= 0

df = df[remove == False]

In [157]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [160]:
print((df['Quantity']<=0).any())

False


In [161]:
print((df['UnitPrice']<=0).any())

True


In [164]:
remove = df["UnitPrice"] <= 0

print("Invalid UnitPrice:", remove.sum())

Invalid UnitPrice: 1166


In [166]:
df = df[remove == False]

In [167]:
print((df['UnitPrice']<=0).any())

False


In [168]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 5226


In [169]:
df = df.drop_duplicates()

In [170]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [171]:
print("Unique customers:", df["CustomerID"].nunique())

Unique customers: 4338


In [172]:
print("Missing CustomerID:", df["CustomerID"].isna().sum())

Missing CustomerID: 132186


In [175]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']

In [176]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,revenue,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,20.34


In [178]:
df["Revenue"].describe()

count    524878.000000
mean         20.275399
std         271.693566
min           0.001000
25%           3.900000
50%           9.920000
75%          17.700000
max      168469.600000
Name: Revenue, dtype: float64

In [179]:
print("Duplicate rows:", df.duplicated().sum())
print("Negative Quantity:", (df["Quantity"] < 0).sum())
print("Zero Quantity:", (df["Quantity"] == 0).sum())
print("Invalid UnitPrice:", (df["UnitPrice"] <= 0).sum())
print("Negative Revenue:", (df["Revenue"] < 0).sum())

Duplicate rows: 0
Negative Quantity: 0
Zero Quantity: 0
Invalid UnitPrice: 0
Negative Revenue: 0


In [181]:
df.to_csv("../data/processed/cleaned_data.csv", index=False)

In [184]:
import os

print(os.path.exists("../data/processed/cleaned_data.csv"))

True
